# E03 Spark Bureau parity smoke

This notebook validates a Spark implementation against the locked pandas
`bureau-v1` contract. It stops before model training and never overwrites the
reference feature block. Only an already aggregated client block is converted
with `toPandas()`.

In [ ]:
# 1. Environment and reusable source
import importlib.util
import json
import os
import platform
import subprocess
import sys
import time
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

OUTPUT_DIR = Path(
    "/kaggle/working/home_credit_outputs/E03_spark_parity_smoke"
).resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
INSTALL_PYSPARK_IF_MISSING = True
REFERENCE_BLOCK_ROOT = os.environ.get("REFERENCE_BUREAU_BLOCK_ROOT")
REFERENCE_BLOCK_NAME = os.environ.get("REFERENCE_BUREAU_BLOCK_NAME", "bureau")
RTOL = 1e-5
ATOL = 1e-6

source_archive = Path("/kaggle/input/e03-spark-source/src.zip")
extracted_source = Path("/kaggle/working/e03-spark-source")
if source_archive.is_file() and not (extracted_source / "src").is_dir():
    extracted_source.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(source_archive) as archive:
        archive.extractall(extracted_source)
discovered_sources = [
    path.parent.parent
    for path in Path("/kaggle/input").glob("**/credit_scoring/__init__.py")
]
source_candidates = [
    Path(os.environ["CREDIT_SCORING_SRC"])
    if os.environ.get("CREDIT_SCORING_SRC")
    else None,
    Path("/kaggle/input/e03-spark-source/src"),
    Path("/kaggle/input/e03-spark-source"),
    extracted_source / "src",
    Path("/kaggle/working/Qaci-datascience/src"),
    Path.cwd() / "src",
    Path.cwd().parent / "src",
    *discovered_sources,
]
for source_candidate in source_candidates:
    if source_candidate and (source_candidate / "credit_scoring").is_dir():
        if str(source_candidate.resolve()) not in sys.path:
            sys.path.insert(0, str(source_candidate.resolve()))
        break

java_check = subprocess.run(
    ["java", "-version"],
    check=False,
    capture_output=True,
    text=True,
)
pyspark_preinstalled = importlib.util.find_spec("pyspark") is not None
if not pyspark_preinstalled and INSTALL_PYSPARK_IF_MISSING:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "pyspark>=3.5,<4"],
        check=True,
    )
if importlib.util.find_spec("pyspark") is None:
    raise RuntimeError("PySpark is unavailable; Spark parity smoke cannot run.")

from pyspark import __version__ as pyspark_version
from pyspark.sql import SparkSession
from pyspark.storagelevel import StorageLevel

from credit_scoring.feature_store import BlockManifest, load_block
from credit_scoring.features.home_credit_bureau import (
    BUILDER_VERSION as PANDAS_BUILDER_VERSION,
)
from credit_scoring.features.home_credit_bureau import build_bureau_features
from credit_scoring.features.home_credit_bureau_spark import (
    BUREAU_EXACT_FEATURE_NAMES,
    BUREAU_FEATURE_FAMILIES,
    BUREAU_FEATURE_NAMES,
    SPARK_BUILDER_VERSION,
    build_bureau_balance_features_spark,
    build_bureau_features_spark,
    bureau_balance_csv_schema_spark,
    bureau_csv_schema_spark,
    write_bureau_feature_block_spark,
)

if PANDAS_BUILDER_VERSION != "bureau-v1":
    raise RuntimeError(f"Unexpected pandas builder: {PANDAS_BUILDER_VERSION}")
if SPARK_BUILDER_VERSION != "bureau-v1-spark-smoke":
    raise RuntimeError(f"Unexpected Spark builder: {SPARK_BUILDER_VERSION}")

print("Python:", platform.python_version())
print("Java available:", java_check.returncode == 0)
print("PySpark preinstalled:", pyspark_preinstalled)
print("PySpark runtime:", pyspark_version)

In [ ]:
# 2. Conservative local Spark session
spark = (
    SparkSession.builder.master("local[2]")
    .appName("E03-bureau-parity-smoke")
    .config("spark.sql.shuffle.partitions", "16")
    .config("spark.default.parallelism", "16")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.execution.arrow.pyspark.enabled", "false")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark application:", spark.sparkContext.applicationId)

In [ ]:
# 3. Deterministic fixture and null-safe parity diagnostics
def make_fixture():
    bureau = pd.DataFrame(
        {
            "SK_ID_CURR": [1, 1, 2, 3],
            "SK_ID_BUREAU": [10, 11, 12, 13],
            "CREDIT_ACTIVE": ["Active", "Closed", "Active", "Closed"],
            "CREDIT_TYPE": ["Credit card", "Car loan", "Mortgage", None],
            "DAYS_CREDIT": [-100, -900, -30, -400],
            "DAYS_CREDIT_ENDDATE": [200.0, -700.0, 100.0, -300.0],
            "CREDIT_DAY_OVERDUE": [0, 12, 0, 0],
            "AMT_CREDIT_SUM": [1000.0, 0.0, 5000.0, 800.0],
            "AMT_CREDIT_SUM_DEBT": [400.0, 100.0, 2500.0, np.nan],
            "AMT_CREDIT_SUM_OVERDUE": [0.0, 50.0, 0.0, np.nan],
            "AMT_CREDIT_SUM_LIMIT": [-10.0, 0.0, 1000.0, np.nan],
            "AMT_CREDIT_MAX_OVERDUE": [0.0, 75.0, np.nan, np.nan],
            "AMT_ANNUITY": [np.nan, np.nan, 300.0, np.nan],
            "CNT_CREDIT_PROLONG": [0, 1, 0, 0],
        }
    )
    balance = pd.DataFrame(
        {
            "SK_ID_BUREAU": [10, 10, 10, 10, 11, 11, 11, 12],
            "MONTHS_BALANCE": [-1, -2, -3, -4, -1, -2, -3, -1],
            "STATUS": ["0", "1", "3", "X", "C", "5", "X", "0"],
        }
    )
    application = pd.DataFrame({"SK_ID_CURR": [1, 2, 3, 4]})
    return bureau, balance, application


def spark_fixture_frames(bureau, balance):
    from pyspark.sql import types as T

    bureau_schema = T.StructType(
        [
            T.StructField("SK_ID_CURR", T.IntegerType(), False),
            T.StructField("SK_ID_BUREAU", T.IntegerType(), False),
            T.StructField("CREDIT_ACTIVE", T.StringType(), True),
            T.StructField("CREDIT_TYPE", T.StringType(), True),
            T.StructField("DAYS_CREDIT", T.IntegerType(), True),
            T.StructField("DAYS_CREDIT_ENDDATE", T.DoubleType(), True),
            T.StructField("CREDIT_DAY_OVERDUE", T.IntegerType(), True),
            T.StructField("AMT_CREDIT_SUM", T.DoubleType(), True),
            T.StructField("AMT_CREDIT_SUM_DEBT", T.DoubleType(), True),
            T.StructField("AMT_CREDIT_SUM_OVERDUE", T.DoubleType(), True),
            T.StructField("AMT_CREDIT_SUM_LIMIT", T.DoubleType(), True),
            T.StructField("AMT_CREDIT_MAX_OVERDUE", T.DoubleType(), True),
            T.StructField("AMT_ANNUITY", T.DoubleType(), True),
            T.StructField("CNT_CREDIT_PROLONG", T.IntegerType(), True),
        ]
    )
    balance_schema = T.StructType(
        [
            T.StructField("SK_ID_BUREAU", T.IntegerType(), False),
            T.StructField("MONTHS_BALANCE", T.IntegerType(), False),
            T.StructField("STATUS", T.StringType(), True),
        ]
    )
    clean_bureau = bureau.astype(object).where(pd.notna(bureau), None)
    spark_bureau = spark.createDataFrame(
        list(clean_bureau.itertuples(index=False, name=None)),
        schema=bureau_schema,
    )
    spark_balance = spark.createDataFrame(
        list(balance.itertuples(index=False, name=None)),
        schema=balance_schema,
    )
    return spark_bureau, spark_balance


def compare_feature_frames(reference, candidate, reference_families, scope):
    reference = reference.sort_values("SK_ID_CURR").reset_index(drop=True)
    candidate = candidate.sort_values("SK_ID_CURR").reset_index(drop=True)
    expected_columns = ["SK_ID_CURR", *BUREAU_FEATURE_NAMES]
    keys = sorted(set(reference["SK_ID_CURR"]).union(candidate["SK_ID_CURR"]))
    left = reference.set_index("SK_ID_CURR").reindex(keys)
    right = candidate.set_index("SK_ID_CURR").reindex(keys)

    schema_rows = []
    for position, feature in enumerate(BUREAU_FEATURE_NAMES, start=1):
        schema_rows.append(
            {
                "scope": scope,
                "position": position,
                "feature": feature,
                "reference_present": feature in reference,
                "candidate_present": feature in candidate,
                "reference_family": reference_families.get(feature),
                "candidate_family": BUREAU_FEATURE_FAMILIES.get(feature),
                "family_match": (
                    reference_families.get(feature)
                    == BUREAU_FEATURE_FAMILIES.get(feature)
                ),
            }
        )

    null_rows = []
    mismatch_rows = []
    first_failing_feature = None
    for feature in BUREAU_FEATURE_NAMES:
        left_values = pd.to_numeric(left[feature], errors="coerce").to_numpy(
            dtype="float64"
        )
        right_values = pd.to_numeric(right[feature], errors="coerce").to_numpy(
            dtype="float64"
        )
        left_null = np.isnan(left_values)
        right_null = np.isnan(right_values)
        null_mismatch = left_null ^ right_null
        both_observed = ~left_null & ~right_null
        differences = np.zeros(len(keys), dtype="float64")
        differences[both_observed] = np.abs(
            left_values[both_observed] - right_values[both_observed]
        )
        if feature in BUREAU_EXACT_FEATURE_NAMES:
            value_mismatch = both_observed & (
                left_values != right_values
            )
            comparison = "exact"
        else:
            value_mismatch = both_observed & ~np.isclose(
                left_values,
                right_values,
                rtol=RTOL,
                atol=ATOL,
            )
            comparison = "tolerance"
        mismatch = null_mismatch | value_mismatch
        mismatch_count = int(mismatch.sum())
        max_absolute_difference = (
            float(differences[both_observed].max())
            if both_observed.any()
            else 0.0
        )
        null_rows.append(
            {
                "scope": scope,
                "feature": feature,
                "reference_null_count": int(left_null.sum()),
                "candidate_null_count": int(right_null.sum()),
                "null_mismatch_count": int(null_mismatch.sum()),
            }
        )
        mismatch_rows.append(
            {
                "scope": scope,
                "feature": feature,
                "comparison": comparison,
                "mismatch_count": mismatch_count,
                "max_absolute_difference": max_absolute_difference,
                "within_tolerance": mismatch_count == 0,
            }
        )
        if mismatch_count and first_failing_feature is None:
            first_failing_feature = feature

    numeric_candidate = candidate.select_dtypes(include="number").to_numpy(
        dtype="float64"
    )
    summary = {
        "scope": scope,
        "row_count_match": len(reference) == len(candidate),
        "reference_row_count": len(reference),
        "candidate_row_count": len(candidate),
        "reference_unique_key_count": int(reference["SK_ID_CURR"].nunique()),
        "candidate_unique_key_count": int(candidate["SK_ID_CURR"].nunique()),
        "candidate_duplicate_key_count": int(
            candidate["SK_ID_CURR"].duplicated().sum()
        ),
        "key_set_match": set(reference["SK_ID_CURR"])
        == set(candidate["SK_ID_CURR"]),
        "feature_name_set_match": set(reference.columns) == set(expected_columns),
        "feature_order_match": list(reference.columns)
        == list(candidate.columns)
        == expected_columns,
        "family_mapping_match": reference_families == BUREAU_FEATURE_FAMILIES,
        "null_policy_match": all(row["null_mismatch_count"] == 0 for row in null_rows),
        "value_parity": all(row["within_tolerance"] for row in mismatch_rows),
        "infinity_count": int(np.isinf(numeric_candidate).sum()),
        "first_failing_feature": first_failing_feature,
    }
    summary["pass"] = all(
        [
            summary["row_count_match"],
            summary["candidate_duplicate_key_count"] == 0,
            summary["key_set_match"],
            summary["feature_name_set_match"],
            summary["feature_order_match"],
            summary["family_mapping_match"],
            summary["null_policy_match"],
            summary["value_parity"],
            summary["infinity_count"] == 0,
        ]
    )
    return (
        summary,
        pd.DataFrame(schema_rows),
        pd.DataFrame(null_rows),
        pd.DataFrame(mismatch_rows),
    )

In [ ]:
# 4. Mandatory synthetic parity assertion before real-data reads
fixture_bureau, fixture_balance, fixture_application = make_fixture()
reference_fixture, reference_fixture_families = build_bureau_features(
    fixture_bureau,
    fixture_balance,
)
spark_fixture_bureau, spark_fixture_balance = spark_fixture_frames(
    fixture_bureau,
    fixture_balance,
)
spark_fixture_balance_features = build_bureau_balance_features_spark(
    spark_fixture_balance
)
spark_fixture_features = build_bureau_features_spark(
    spark_fixture_bureau,
    spark_fixture_balance_features,
)
candidate_fixture = spark_fixture_features.toPandas()
(
    fixture_summary,
    fixture_schema,
    fixture_nulls,
    fixture_mismatches,
) = compare_feature_frames(
    reference_fixture,
    candidate_fixture,
    reference_fixture_families,
    "deterministic_fixture",
)
fixture_merged = fixture_application.merge(
    candidate_fixture,
    on="SK_ID_CURR",
    how="left",
    validate="one_to_one",
)
fixture_application_row_count_unchanged = len(fixture_merged) == len(
    fixture_application
)
if not fixture_summary["pass"]:
    raise AssertionError(
        "Synthetic Spark/pandas parity failed at "
        f"{fixture_summary['first_failing_feature']}"
    )
if not fixture_application_row_count_unchanged:
    raise AssertionError("Synthetic application merge changed row count")
print("Synthetic Spark/pandas parity: PASS")

In [ ]:
# 5. Read full Kaggle Bureau CSV files using explicit Spark schemas
data_candidates = [
    Path("/kaggle/input/home-credit-default-risk"),
    Path("/kaggle/input/competitions/home-credit-default-risk"),
]
DATA_DIR = next(
    (
        path
        for path in data_candidates
        if (path / "bureau.csv").is_file()
        and (path / "bureau_balance.csv").is_file()
    ),
    None,
)
if DATA_DIR is None:
    raise FileNotFoundError(f"Home Credit data not found in {data_candidates}")

raw_bureau = (
    spark.read.option("header", True)
    .option("mode", "FAILFAST")
    .schema(bureau_csv_schema_spark())
    .csv(str(DATA_DIR / "bureau.csv"))
)
raw_bureau_balance = (
    spark.read.option("header", True)
    .option("mode", "FAILFAST")
    .schema(bureau_balance_csv_schema_spark())
    .csv(str(DATA_DIR / "bureau_balance.csv"))
)
application_ids = pd.concat(
    [
        pd.read_csv(DATA_DIR / "application_train.csv", usecols=["SK_ID_CURR"]),
        pd.read_csv(DATA_DIR / "application_test.csv", usecols=["SK_ID_CURR"]),
    ],
    ignore_index=True,
)
print("Data directory:", DATA_DIR)

In [ ]:
# 6. Build and write the full Spark candidate block
aggregation_started = time.perf_counter()
full_balance_features = build_bureau_balance_features_spark(raw_bureau_balance)
full_spark_features = build_bureau_features_spark(
    raw_bureau,
    full_balance_features,
).persist(StorageLevel.MEMORY_AND_DISK)
candidate_row_count = full_spark_features.count()
aggregation_runtime_seconds = time.perf_counter() - aggregation_started

candidate_manifest = write_bureau_feature_block_spark(
    full_spark_features,
    root=OUTPUT_DIR,
)
candidate_features = full_spark_features.toPandas()
application_merged = application_ids.merge(
    candidate_features,
    on="SK_ID_CURR",
    how="left",
    validate="one_to_one",
)
application_row_count_unchanged = len(application_merged) == len(application_ids)

aggregation_diagnostics = {
    "bureau_row_count": raw_bureau.count(),
    "bureau_unique_loan_count": raw_bureau.select("SK_ID_BUREAU").distinct().count(),
    "bureau_balance_row_count": raw_bureau_balance.count(),
    "bureau_balance_unique_loan_count": raw_bureau_balance.select(
        "SK_ID_BUREAU"
    ).distinct().count(),
    "balance_feature_row_count": full_balance_features.count(),
    "candidate_row_count": candidate_row_count,
    "candidate_unique_key_count": int(candidate_features["SK_ID_CURR"].nunique()),
    "candidate_duplicate_key_count": int(
        candidate_features["SK_ID_CURR"].duplicated().sum()
    ),
    "application_row_count": len(application_ids),
    "application_row_count_after_merge": len(application_merged),
    "application_row_count_unchanged": application_row_count_unchanged,
    "aggregation_runtime_seconds": aggregation_runtime_seconds,
}
(OUTPUT_DIR / "aggregation_diagnostics.json").write_text(
    json.dumps(aggregation_diagnostics, indent=2),
    encoding="utf-8",
)

In [ ]:
# 7. Optional full reference comparison and required diagnostics
if REFERENCE_BLOCK_ROOT is None:
    discovered = sorted(
        Path("/kaggle/input").glob(f"**/{REFERENCE_BLOCK_NAME}.manifest.json")
    )
    if discovered:
        REFERENCE_BLOCK_ROOT = str(discovered[0].parent)

full_summary = None
full_schema = pd.DataFrame()
full_nulls = pd.DataFrame()
full_mismatches = pd.DataFrame()
if REFERENCE_BLOCK_ROOT:
    reference_block = load_block(
        REFERENCE_BLOCK_NAME,
        root=REFERENCE_BLOCK_ROOT,
        expected_builder_version=PANDAS_BUILDER_VERSION,
    )
    (
        full_summary,
        full_schema,
        full_nulls,
        full_mismatches,
    ) = compare_feature_frames(
        reference_block.frame,
        candidate_features,
        dict(reference_block.manifest.families),
        "full_block",
    )
    full_reference_status = "PASS" if full_summary["pass"] else "FAIL"
else:
    full_reference_status = "SKIPPED_REFERENCE_PATH_NOT_PROVIDED"

manifest_payload = json.loads(
    (OUTPUT_DIR / "bureau_spark_candidate.manifest.json").read_text(
        encoding="utf-8"
    )
)
round_trip_manifest = BlockManifest.from_dict(manifest_payload)
manifest_valid = all(
    [
        round_trip_manifest.builder_version == SPARK_BUILDER_VERSION,
        round_trip_manifest.key_column == "SK_ID_CURR",
        round_trip_manifest.feature_names == BUREAU_FEATURE_NAMES,
        dict(round_trip_manifest.families) == BUREAU_FEATURE_FAMILIES,
        round_trip_manifest.row_count == candidate_row_count,
        round_trip_manifest.unique_key_count == candidate_row_count,
    ]
)

schema_comparison = pd.concat(
    [frame for frame in [fixture_schema, full_schema] if not frame.empty],
    ignore_index=True,
)
null_comparison = pd.concat(
    [frame for frame in [fixture_nulls, full_nulls] if not frame.empty],
    ignore_index=True,
)
value_mismatch_summary = pd.concat(
    [frame for frame in [fixture_mismatches, full_mismatches] if not frame.empty],
    ignore_index=True,
)
schema_comparison.to_csv(OUTPUT_DIR / "schema_comparison.csv", index=False)
null_comparison.to_csv(OUTPUT_DIR / "null_comparison.csv", index=False)
value_mismatch_summary.to_csv(
    OUTPUT_DIR / "value_mismatch_summary.csv",
    index=False,
)

overall_pass = all(
    [
        fixture_summary["pass"],
        fixture_application_row_count_unchanged,
        candidate_row_count == int(candidate_features["SK_ID_CURR"].nunique()),
        not candidate_features["SK_ID_CURR"].duplicated().any(),
        list(candidate_features.columns) == ["SK_ID_CURR", *BUREAU_FEATURE_NAMES],
        application_row_count_unchanged,
        manifest_valid,
        full_summary is None or full_summary["pass"],
    ]
)
first_failing_feature = (
    fixture_summary["first_failing_feature"]
    or (full_summary or {}).get("first_failing_feature")
)
parity_summary = {
    "status": "PASS" if overall_pass else "FAIL",
    "pandas_builder_version": PANDAS_BUILDER_VERSION,
    "spark_builder_version": SPARK_BUILDER_VERSION,
    "rtol": RTOL,
    "atol": ATOL,
    "fixture": fixture_summary,
    "full_reference_status": full_reference_status,
    "full_reference": full_summary,
    "manifest_valid": manifest_valid,
    "application_row_count_unchanged": application_row_count_unchanged,
    "first_failing_feature": first_failing_feature,
    "safe_to_promote": bool(full_summary and full_summary["pass"] and overall_pass),
}
(OUTPUT_DIR / "parity_summary.json").write_text(
    json.dumps(parity_summary, indent=2),
    encoding="utf-8",
)

spark_environment = {
    "python": platform.python_version(),
    "java_available": java_check.returncode == 0,
    "java_version_output": (java_check.stderr or java_check.stdout).strip(),
    "pyspark_preinstalled": pyspark_preinstalled,
    "pyspark_version": pyspark_version,
    "spark_version": spark.version,
    "spark_application_id": spark.sparkContext.applicationId,
}
(OUTPUT_DIR / "spark_environment.json").write_text(
    json.dumps(spark_environment, indent=2),
    encoding="utf-8",
)

print(json.dumps(parity_summary, indent=2))
if not overall_pass:
    raise AssertionError(
        "Spark parity smoke failed; first feature: "
        f"{first_failing_feature or 'structural criterion'}"
    )

In [ ]:
# 8. Stop before LightGBM
full_spark_features.unpersist()
spark.stop()
print("Spark parity smoke completed. No model training was started.")

## Interpretation

`status=PASS` proves deterministic parity and structural safety. Promotion is
still blocked unless `full_reference_status=PASS`; a missing reference block is
recorded as a skip, never silently treated as full parity.